# 第6章　ディープラーニングエンジニアリングの要諦 ― 最適化・初期化・正規化・正則化

**『本格実装 医療診断支援AI（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 6.2　最適化器 ― AdamWを既定に置く

In [ ]:
import torch

def param_groups(model, wd=1e-2):
    """重み減衰をかける層と、かけない層（正規化層の係数・バイアス）を分ける。"""
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.ndim == 1 or name.endswith(".bias"):   # BN/GNの係数とバイアスは1次元
            no_decay.append(p)
        else:
            decay.append(p)
    return [{"params": decay,    "weight_decay": wd},
            {"params": no_decay, "weight_decay": 0.0}]

optimizer = torch.optim.AdamW(param_groups(model, wd=1e-2), lr=3e-4)

## 6.3　学習率 ― ウォームアップとコサイン減衰

In [ ]:
import math
from torch.optim.lr_scheduler import LambdaLR

def cosine_with_warmup(optimizer, warmup_steps, total_steps, min_ratio=0.01):
    def factor(step):
        warm = min((step + 1) / max(1, warmup_steps), 1.0)      # 1ステップ目から線形に立ち上げる
        prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        prog = min(max(prog, 0.0), 1.0)                          # 0〜1に収める
        cos = 0.5 * (1.0 + math.cos(math.pi * prog))             # 1→0へなめらかに減衰
        return warm * (min_ratio + (1.0 - min_ratio) * cos)
    return LambdaLR(optimizer, lr_lambda=factor)

total_steps = len(train_loader) * epochs
scheduler = cosine_with_warmup(optimizer, warmup_steps=int(0.05 * total_steps),
                               total_steps=total_steps)
# 学習ループ内では、optimizer.step() の直後に scheduler.step() を呼ぶ（1ステップごと）

## 6.4　初期化と正規化層 ― 小バッチの3D学習で何を選ぶか

In [ ]:
import math
import torch.nn as nn

def init_rare_head(conv, prior=0.01):
    """陽性がまれなタスクの出力層（sigmoid 出力用。softmax には使えない ― 本文参照）。
    最初から「ほぼ陰性」と答える状態から始める。"""
    nn.init.normal_(conv.weight, std=0.01)
    nn.init.constant_(conv.bias, -math.log((1.0 - prior) / prior))  # prior=0.01 なら約 -4.6

## 6.5　勾配を守る ― 消失・爆発・クリッピング・混合精度のNaN

In [ ]:
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
optimizer.step()
optimizer.zero_grad(set_to_none=True)

In [ ]:
scaler = torch.amp.GradScaler()
for x, y in train_loader:
    with torch.autocast("cuda", dtype=torch.float16):
        loss = criterion(model(x), y)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)                                   # 先に実スケールへ戻す
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)      # そのうえでクリップ
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    scheduler.step()

## 6.7　正則化の道具箱

In [ ]:
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

ema = AveragedModel(model, multi_avg_fn=get_ema_multi_avg_fn(0.999))
# 学習ループ内、optimizer.step() の後に：
#     ema.update_parameters(model)
# 検証・提出・保存には、生のmodelではなく ema.module を使う
# BatchNormを含む場合、統計（buffers）は既定（use_buffers=False）では作成時の複製のまま更新されない（本例）。平均する方法（use_buffers=True）、
# 学習データで再推定する方法もあり、採用した方法と検証条件を記録する